<a href="https://colab.research.google.com/github/aditisinghxd/RAG-System-with-LangChain-and-FastAPI/blob/main/RAG_System_with_LangChain_and_FastAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import platform

print("Python:", sys.version)
print("System:", platform.system())
print("Machine:", platform.machine())

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
System: Linux
Machine: x86_64


In [2]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-openai faiss-cpu python-dotenv

In [3]:
import langchain
import langchain_community
import langchain_openai
import langchain_text_splitters
import faiss

from dotenv import load_dotenv

print("LangChain:", langchain.__version__)
print("FAISS imported successfully")
print("All required packages are working ✅")

/tmp/ipykernel_16863/336641878.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


LangChain: 1.3.17
FAISS imported successfully
All required packages are working ✅


In [4]:
import os

os.makedirs("data", exist_ok=True)

polar_bear_text = """
Polar bears live mainly in Arctic regions.

They rely heavily on sea ice because they use it as a platform
for hunting seals.

Polar bears have several adaptations for cold environments,
including thick fur and a layer of body fat.

Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

Protecting Arctic habitats is therefore important for
polar bear conservation.
"""

with open("data/my_document.txt", "w", encoding="utf-8") as file:
    file.write(polar_bear_text)

print("Document created!")

Document created!


In [5]:
with open("data/my_document.txt", "r", encoding="utf-8") as file:
    content = file.read()

print(content)


Polar bears live mainly in Arctic regions.

They rely heavily on sea ice because they use it as a platform
for hunting seals.

Polar bears have several adaptations for cold environments,
including thick fur and a layer of body fat.

Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

Protecting Arctic habitats is therefore important for
polar bear conservation.



In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/my_document.txt", encoding="utf-8")

documents = loader.load()

print(documents)

[Document(metadata={'source': 'data/my_document.txt'}, page_content='\nPolar bears live mainly in Arctic regions.\n\nThey rely heavily on sea ice because they use it as a platform\nfor hunting seals.\n\nPolar bears have several adaptations for cold environments,\nincluding thick fur and a layer of body fat.\n\nClimate change is causing Arctic sea ice to decline.\nThis reduces the amount of time polar bears have available\nfor hunting and can make it harder for them to obtain enough food.\n\nProtecting Arctic habitats is therefore important for\npolar bear conservation.\n')]


In [7]:
len(documents)

1

In [8]:
print(documents[0].page_content)


Polar bears live mainly in Arctic regions.

They rely heavily on sea ice because they use it as a platform
for hunting seals.

Polar bears have several adaptations for cold environments,
including thick fur and a layer of body fat.

Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

Protecting Arctic habitats is therefore important for
polar bear conservation.



In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30
)

In [11]:
document_chunks = splitter.split_documents(documents)

In [12]:
print("Number of chunks:", len(document_chunks))

Number of chunks: 4


In [13]:
for i, chunk in enumerate(document_chunks):
    print("\n--- CHUNK", i + 1, "---")
    print(chunk.page_content)


--- CHUNK 1 ---
Polar bears live mainly in Arctic regions.

They rely heavily on sea ice because they use it as a platform
for hunting seals.

--- CHUNK 2 ---
Polar bears have several adaptations for cold environments,
including thick fur and a layer of body fat.

--- CHUNK 3 ---
Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

--- CHUNK 4 ---
Protecting Arctic habitats is therefore important for
polar bear conservation.


In [14]:
for i, chunk in enumerate(document_chunks):
    print(f"Chunk {i + 1}")
    print("Length:", len(chunk.page_content))
    print("Metadata:", chunk.metadata)
    print()

Chunk 1
Length: 125
Metadata: {'source': 'data/my_document.txt'}

Chunk 2
Length: 104
Metadata: {'source': 'data/my_document.txt'}

Chunk 3
Length: 178
Metadata: {'source': 'data/my_document.txt'}

Chunk 4
Length: 78
Metadata: {'source': 'data/my_document.txt'}



In [15]:
%pip install -q langchain-huggingface sentence-transformers

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully ✅


In [17]:
test_vector = embeddings.embed_query(
    "Polar bears depend on sea ice."
)

print(type(test_vector))
print("Vector length:", len(test_vector))
print("First 10 numbers:")
print(test_vector[:10])

<class 'list'>
Vector length: 384
First 10 numbers:
[-0.028995594009757042, 0.0046784537844359875, 0.06314288824796677, 0.0602874755859375, 0.00865908619016409, 0.04066053405404091, 0.004770329687744379, 0.018250739201903343, -0.005906157661229372, 0.043514613062143326]


In [18]:
import numpy as np

sentence_a = "Polar bears depend on sea ice."
sentence_b = "Sea ice is important for polar bears."
sentence_c = "I enjoy eating chocolate cake."

vector_a = embeddings.embed_query(sentence_a)
vector_b = embeddings.embed_query(sentence_b)
vector_c = embeddings.embed_query(sentence_c)

In [19]:
def cosine_similarity(vector1, vector2):
    vector1 = np.array(vector1)
    vector2 = np.array(vector2)

    return np.dot(vector1, vector2) / (
        np.linalg.norm(vector1) * np.linalg.norm(vector2)
    )


similarity_ab = cosine_similarity(vector_a, vector_b)
similarity_ac = cosine_similarity(vector_a, vector_c)

print("A vs B:", similarity_ab)
print("A vs C:", similarity_ac)

A vs B: 0.9020547071977326
A vs C: 0.05040178836986733


In [20]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    document_chunks,
    embeddings
)

print("FAISS vector store created ✅")
print("Vectors stored:", vector_store.index.ntotal)

FAISS vector store created ✅
Vectors stored: 4


In [21]:
query = "What problems are polar bears facing?"

results = vector_store.similarity_search(
    query,
    k=2
)

print("Number of results:", len(results))

for i, result in enumerate(results):
    print(f"\n--- RESULT {i + 1} ---")
    print(result.page_content)

Number of results: 2

--- RESULT 1 ---
Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

--- RESULT 2 ---
Protecting Arctic habitats is therefore important for
polar bear conservation.


In [22]:
scored_results = vector_store.similarity_search_with_score(
    query,
    k=4
)

for i, (document, score) in enumerate(scored_results):
    print(f"\n--- RESULT {i + 1} ---")
    print("Score:", score)
    print(document.page_content)


--- RESULT 1 ---
Score: 0.710942
Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

--- RESULT 2 ---
Score: 0.75751036
Protecting Arctic habitats is therefore important for
polar bear conservation.

--- RESULT 3 ---
Score: 0.8063276
Polar bears live mainly in Arctic regions.

They rely heavily on sea ice because they use it as a platform
for hunting seals.

--- RESULT 4 ---
Score: 0.83300185
Polar bears have several adaptations for cold environments,
including thick fur and a layer of body fat.


In [23]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

print("Retriever created ✅")

Retriever created ✅


In [24]:
retrieved_docs = retriever.invoke(
    "What problems are polar bears facing?"
)

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- DOCUMENT {i + 1} ---")
    print(doc.page_content)


--- DOCUMENT 1 ---
Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

--- DOCUMENT 2 ---
Protecting Arctic habitats is therefore important for
polar bear conservation.


In [25]:
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

print(context)

Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

Protecting Arctic habitats is therefore important for
polar bear conservation.


In [26]:
question = "What problems are polar bears facing?"

prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)


Use the context below to answer the question.

Context:
Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

Protecting Arctic habitats is therefore important for
polar bear conservation.

Question:
What problems are polar bears facing?

Answer:



In [27]:
%pip install -q transformers sentencepiece accelerate

In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Free LLM loaded successfully ✅")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Free LLM loaded successfully ✅


In [29]:
rag_inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

rag_output_ids = model.generate(
    **rag_inputs,
    max_new_tokens=50,
    do_sample=False
)

rag_answer = tokenizer.decode(
    rag_output_ids[0],
    skip_special_tokens=True
)

print("RAG Answer:", rag_answer)

RAG Answer: (iii).


In [30]:
def ask_rag(question):

    # 1. Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # 2. Combine retrieved documents into context
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # 3. Build the prompt
    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    # 4. Convert prompt into tokens
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # 5. Generate an answer
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    # 6. Convert generated tokens back to text
    answer = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
    )

    return answer, retrieved_docs

In [31]:
answer, docs = ask_rag(
    "Why is climate change a problem for polar bears?"
)

print("GENERATED ANSWER:")
print(answer)

print("\nRETRIEVED DOCUMENTS:")

for i, doc in enumerate(docs):
    print(f"\n--- DOCUMENT {i + 1} ---")
    print(doc.page_content)

GENERATED ANSWER:
a).

RETRIEVED DOCUMENTS:

--- DOCUMENT 1 ---
Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

--- DOCUMENT 2 ---
Protecting Arctic habitats is therefore important for
polar bear conservation.


In [32]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 Base loaded successfully ✅")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 Base loaded successfully ✅


In [33]:
answer, docs = ask_rag(
    "Why is climate change a problem for polar bears?"
)

In [34]:
print("GENERATED ANSWER:")
print(answer)

print("\nRETRIEVED DOCUMENTS:")

for i, doc in enumerate(docs):
    print(f"\n--- DOCUMENT {i + 1} ---")
    print(doc.page_content)

GENERATED ANSWER:
This reduces the amount of time polar bears have available for hunting and can make it harder for them to obtain enough food

RETRIEVED DOCUMENTS:

--- DOCUMENT 1 ---
Climate change is causing Arctic sea ice to decline.
This reduces the amount of time polar bears have available
for hunting and can make it harder for them to obtain enough food.

--- DOCUMENT 2 ---
Protecting Arctic habitats is therefore important for
polar bear conservation.


In [35]:
%pip install -q fastapi uvicorn

In [59]:
%%writefile test_main.py

from fastapi.testclient import TestClient
from main import app

client = TestClient(app)


def test_home():
    response = client.get("/")

    assert response.status_code == 200
    assert response.json() == {
        "message": "RAG API is running"
    }


def test_query():
    question = "Why is climate change a problem for polar bears?"

    response = client.get(
        "/query",
        params={"question": question}
    )

    assert response.status_code == 200

    data = response.json()

    assert data["question"] == question
    assert isinstance(data["answer"], str)
    assert len(data["answer"]) > 0

    # The temporary rag.py must be replaced
    assert not data["answer"].startswith("Received question:")

Overwriting test_main.py


In [60]:
!pytest -q test_main.py

.F                                                                       [100%]
=================================== FAILURES ===================================
__________________________________ test_query __________________________________

    def test_query():
        question = "Why is climate change a problem for polar bears?"
    
        response = client.get(
            "/query",
            params={"question": question}
        )
    
        assert response.status_code == 200
    
        data = response.json()
    
        assert data["question"] == question
        assert isinstance(data["answer"], str)
        assert len(data["answer"]) > 0
    
        # The temporary rag.py must be replaced
>       assert not data["answer"].startswith("Received question:")
E       AssertionError: assert not True
E        +  where True = <built-in method startswith of str object at 0x7f948a6d2100>('Received question:')
E        +    where <built-in method startswith of str object at 0x7

In [57]:
%%writefile main.py

from fastapi import FastAPI
from rag import ask_rag

app = FastAPI()


@app.get("/")
def home():
    return {"message": "RAG API is running"}


@app.get("/query")
def query_rag(question: str):

    answer = ask_rag(question)

    return {
        "question": question,
        "answer": answer
    }

Overwriting main.py


In [58]:
!pytest -q test_main.py

..                                                                       [100%]
2 passed in 0.53s


In [61]:
%%writefile rag.py

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ---------------------------------
# 1. Load the document
# ---------------------------------

loader = TextLoader(
    "data/my_document.txt",
    encoding="utf-8"
)

documents = loader.load()


# ---------------------------------
# 2. Split document into chunks
# ---------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30
)

document_chunks = splitter.split_documents(documents)


# ---------------------------------
# 3. Create free embeddings
# ---------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# ---------------------------------
# 4. Store embeddings in FAISS
# ---------------------------------

vector_store = FAISS.from_documents(
    document_chunks,
    embeddings
)


# ---------------------------------
# 5. Create retriever
# ---------------------------------

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)


# ---------------------------------
# 6. Load free LLM
# ---------------------------------

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# ---------------------------------
# 7. Complete RAG function
# ---------------------------------

def ask_rag(question: str):

    # Retrieve relevant document chunks
    retrieved_docs = retriever.invoke(question)

    # Combine chunks into context
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Build prompt
    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    # Convert prompt into model tokens
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Generate answer
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    # Decode generated tokens
    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return answer

Overwriting rag.py


In [62]:
!pytest -q test_main.py

..                                                                       [100%]
2 passed in 28.86s


In [63]:
import subprocess
import time

server = subprocess.Popen(
    [
        "uvicorn",
        "main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ]
)

time.sleep(3)

print("FastAPI server started")

FastAPI server started


In [64]:
import requests

response = requests.get("http://127.0.0.1:8000/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'RAG API is running'}


In [65]:
question = "Why is climate change a problem for polar bears?"

response = requests.get(
    "http://127.0.0.1:8000/query",
    params={"question": question}
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'question': 'Why is climate change a problem for polar bears?', 'answer': 'This reduces the amount of time polar bears have available for hunting and can make it harder for them to obtain enough food'}


In [68]:
from google.colab import output

docs_url = output.eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("Open this URL:")
print(docs_url + "/docs")

Open this URL:
https://8000-m-s-kkb-use1c0-1yylf3ygxl5l6-c.us-east1-0.prod.colab.dev/docs
